# 📊 Egypt Job Market Analysis
---
This notebook loads scraped job data from the **Supabase PostgreSQL** database,
performs comprehensive exploratory data analysis, and saves a local CSV snapshot for monitoring.

**Data Source:** Wuzzuf.net via automated Scrapy pipeline  
**Database:** Supabase PostgreSQL (IPv4 connection pooler)  
**Pipeline:** NLP skill extraction → UPSERT → SQL Views

## 1. Setup & Database Connection

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import psycopg2
from datetime import datetime

# Style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Load .env for local development
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_URL = os.environ.get('DATABASE_URL')
assert DB_URL, 'Set DATABASE_URL in your .env or environment variables'
print('✅ DATABASE_URL loaded')

## 2. Load Data from Supabase

In [ ]:
conn = psycopg2.connect(DB_URL)

df = pd.read_sql("""
    SELECT title, company, location, experience, job_type,
           salary, career_level, date_posted, keywords,
           description, url, skills,
           date_first_seen, date_last_seen, status
    FROM job_postings
    WHERE status = 'active'
    ORDER BY date_first_seen DESC;
""", conn)

conn.close()

print(f'✅ Loaded {len(df):,} active job postings from Supabase')
print(f'   Unique companies : {df["company"].nunique():,}')
print(f'   Unique locations : {df["location"].nunique():,}')
print(f'   Date range       : {df["date_first_seen"].min()} → {df["date_first_seen"].max()}')

## 3. Save Local CSV Snapshot

In [ ]:
os.makedirs('output', exist_ok=True)
today = datetime.utcnow().strftime('%Y-%m-%d')
csv_path = f'output/job_market_snapshot_{today}.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'💾 CSV saved: {csv_path} ({len(df):,} rows)')

## 4. Dataset Overview & Data Quality

In [ ]:
df.info()

In [ ]:
df.head(5)

In [ ]:
# Missing / empty data audit
quality = pd.DataFrame({
    'Total': len(df),
    'Non-Null': df.notna().sum(),
    'Empty Strings': df.apply(lambda c: (c == '').sum()),
    'Useful (%)': round(
        (df.notna().sum() - df.apply(lambda c: (c == '').sum())) / len(df) * 100, 1
    )
})
quality.style.background_gradient(subset=['Useful (%)'], cmap='RdYlGn')

## 5. Top Hiring Companies

In [ ]:
top_n = 20
company_counts = df['company'].value_counts().head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('viridis', top_n)
company_counts.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Number of Open Positions')
ax.set_title(f'Top {top_n} Hiring Companies in Egypt', fontsize=16, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(company_counts.values):
    ax.text(v + 0.3, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('output/top_companies.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Most In-Demand Skills

In [ ]:
# Explode the skills array column
skills_series = df['skills'].dropna().explode()
skills_series = skills_series[skills_series != '']
skill_counts = skills_series.value_counts()

top_n_skills = 25
top_skills = skill_counts.head(top_n_skills)

fig, ax = plt.subplots(figsize=(14, 9))
colors = sns.color_palette('magma', top_n_skills)
top_skills.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Frequency (number of job postings)')
ax.set_title(f'Top {top_n_skills} Most Demanded Skills in Egypt\'s Job Market',
             fontsize=16, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(top_skills.values):
    pct = v / len(df) * 100
    ax.text(v + 0.3, i, f'{v} ({pct:.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('output/top_skills.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal unique skills detected: {skill_counts.nunique()}')
print(f'Average skills per job: {df["skills"].dropna().apply(len).mean():.1f}')

## 7. Location Distribution

In [ ]:
loc_df = df[df['location'].notna() & (df['location'] != '')]
loc_counts = loc_df['location'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
loc_counts.plot(kind='bar', ax=ax, color=sns.color_palette('coolwarm', len(loc_counts)))
ax.set_ylabel('Number of Jobs')
ax.set_title('Job Distribution by Location', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('output/location_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Career Level Distribution

In [ ]:
career_df = df[df['career_level'].notna() & (df['career_level'] != '')]

if not career_df.empty:
    level_counts = career_df['career_level'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Pie chart
    level_counts.plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                      colors=sns.color_palette('Set2', len(level_counts)),
                      startangle=90)
    axes[0].set_ylabel('')
    axes[0].set_title('Career Level Split', fontsize=14, fontweight='bold')

    # Bar chart
    level_counts.plot(kind='bar', ax=axes[1],
                      color=sns.color_palette('Set2', len(level_counts)))
    axes[1].set_ylabel('Count')
    axes[1].set_title('Career Level Counts', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig('output/career_level.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ No career level data available yet.')

## 9. Job Type Breakdown

In [ ]:
type_df = df[df['job_type'].notna() & (df['job_type'] != '')]

if not type_df.empty:
    type_counts = type_df['job_type'].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 6))
    type_counts.plot(kind='bar', ax=ax, color=sns.color_palette('pastel', len(type_counts)))
    ax.set_ylabel('Count')
    ax.set_title('Job Type Breakdown', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    for i, v in enumerate(type_counts.values):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('output/job_type_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ No job type data available yet.')

## 10. Salary Insights

In [ ]:
salary_df = df[
    df['salary'].notna() &
    ~df['salary'].isin(['', 'Not Specified', 'Confidential'])
]

total = len(df)
with_salary = len(salary_df)
print(f'Jobs with salary info: {with_salary:,} / {total:,} ({with_salary/total*100:.1f}%)')
print(f'Jobs without salary  : {total - with_salary:,}\n')

if not salary_df.empty:
    print('Sample salary ranges:')
    display(salary_df[['title', 'company', 'salary']].head(15))

## 11. Hiring Trends Over Time

In [ ]:
df['date_first_seen'] = pd.to_datetime(df['date_first_seen'])
daily = df.groupby(df['date_first_seen'].dt.date).size()
cumulative = daily.cumsum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Daily new jobs
ax1.bar(daily.index, daily.values, color='#e74c3c', alpha=0.8)
ax1.set_title('Daily New Job Postings', fontsize=14, fontweight='bold')
ax1.set_ylabel('New Jobs')
ax1.grid(True, alpha=0.3)

# Cumulative
ax2.plot(cumulative.index, cumulative.values, color='#2ecc71', linewidth=2.5)
ax2.fill_between(cumulative.index, cumulative.values, alpha=0.3, color='#2ecc71')
ax2.set_title('Cumulative Job Postings Over Time', fontsize=14, fontweight='bold')
ax2.set_ylabel('Total Jobs')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/hiring_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'First scrape  : {daily.index[0]}')
print(f'Latest scrape : {daily.index[-1]}')
print(f'Avg jobs/day  : {daily.mean():.1f}')

## 12. Skill Co-occurrence Heatmap
Which skills are most commonly required **together** in the same job posting?

In [ ]:
# Filter jobs with at least 2 skills
multi_skill = df['skills'].dropna()
multi_skill = multi_skill[multi_skill.apply(lambda x: isinstance(x, list) and len(x) >= 2)]

if not multi_skill.empty:
    # Get top 15 skills for the heatmap
    all_skills = multi_skill.explode()
    top_15 = all_skills.value_counts().head(15).index.tolist()

    # Build co-occurrence matrix
    matrix = pd.DataFrame(0, index=top_15, columns=top_15)
    for skills_list in multi_skill:
        filtered = [s for s in skills_list if s in top_15]
        for i, s1 in enumerate(filtered):
            for s2 in filtered[i+1:]:
                matrix.loc[s1, s2] += 1
                matrix.loc[s2, s1] += 1

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                square=True, linewidths=0.5, cbar_kws={'label': 'Co-occurrence Count'})
    ax.set_title('Skill Co-occurrence Heatmap (Top 15 Skills)',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('output/skill_cooccurrence.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ Not enough multi-skill data for co-occurrence analysis.')

## 13. Experience Level Analysis

In [ ]:
exp_df = df[df['experience'].notna() & (df['experience'] != '')]

if not exp_df.empty:
    exp_counts = exp_df['experience'].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 6))
    exp_counts.plot(kind='barh', ax=ax, color=sns.color_palette('crest', len(exp_counts)))
    ax.set_xlabel('Number of Jobs')
    ax.set_title('Experience Level Requirements', fontsize=16, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('output/experience_levels.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ No experience data available yet.')

## 14. Skills Demand by Career Level
Which skills are more demanded at junior vs senior levels?

In [ ]:
career_skills = df[
    df['career_level'].notna() &
    (df['career_level'] != '') &
    df['skills'].notna()
].copy()

if not career_skills.empty:
    # Explode skills
    cs_exploded = career_skills.explode('skills')
    cs_exploded = cs_exploded[cs_exploded['skills'] != '']

    # Get top 10 skills overall
    top10 = cs_exploded['skills'].value_counts().head(10).index.tolist()
    cs_filtered = cs_exploded[cs_exploded['skills'].isin(top10)]

    # Crosstab
    ct = pd.crosstab(cs_filtered['skills'], cs_filtered['career_level'])

    fig, ax = plt.subplots(figsize=(14, 8))
    ct.plot(kind='bar', ax=ax, colormap='Set2')
    ax.set_title('Top 10 Skills Demand by Career Level', fontsize=16, fontweight='bold')
    ax.set_ylabel('Number of Jobs')
    ax.set_xlabel('Skill')
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Career Level', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('output/skills_by_career_level.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ Not enough data for career level × skill analysis.')

## 15. Summary Statistics

In [ ]:
print('=' * 60)
print('📊 ANALYSIS COMPLETE — KEY METRICS')
print('=' * 60)
print(f'  Total active jobs      : {len(df):,}')
print(f'  Unique companies       : {df["company"].nunique():,}')
print(f'  Unique locations       : {df["location"].nunique():,}')
print(f'  Unique skills detected : {skill_counts.nunique() if "skill_counts" in dir() else "N/A"}')
print(f'  With salary info       : {len(salary_df):,} ({len(salary_df)/len(df)*100:.1f}%)')
print(f'  With career level      : {len(career_df):,} ({len(career_df)/len(df)*100:.1f}%)')
print(f'  Local CSV              : {csv_path}')
print('=' * 60)
print('\n📁 All charts saved to output/ directory.')

## 16. Job Role Clustering (K-Means)
Using TF-IDF and K-Means to find natural groupings of jobs based on the skills they require.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Prepare data for clustering
cluster_df = df[df['skills'].notna() & (df['skills'].apply(lambda x: isinstance(x, list) and len(x) > 0))].copy()
cluster_df['skills_str'] = cluster_df['skills'].apply(lambda x: ' '.join(x))

if not cluster_df.empty and len(cluster_df) > 50:
    # TF-IDF Vectorization
    vectorizer = TfidfVectorizer(max_features=100)
    X = vectorizer.fit_transform(cluster_df['skills_str'])
    
    # K-Means Clustering
    num_clusters = 5
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_df['cluster'] = kmeans.fit_predict(X)
    
    # Get top skills per cluster
    order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
    terms = vectorizer.get_feature_names_out()
    
    print("Top skills per Job Persona (Cluster):")
    for i in range(num_clusters):
        top_skills = [terms[ind] for ind in order_centroids[i, :7]]
        print(f"Persona {i}: {', '.join(top_skills)}")
        
    # PCA for 2D Visualization
    pca = PCA(n_components=2, random_state=42)
    pca_result = pca.fit_transform(X.toarray())
    cluster_df['pca_1'] = pca_result[:, 0]
    cluster_df['pca_2'] = pca_result[:, 1]
    
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='pca_1', y='pca_2', hue='cluster', palette='tab10', \
                    data=cluster_df, alpha=0.7, s=50)
    plt.title('Job Role Clusters (PCA on Skills)', fontsize=16, fontweight='bold')
    plt.savefig('output/job_clusters_pca.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️ Not enough data with skills for clustering.')

## 17. Company Segmentation
Segmenting top companies based on their hiring volume and preferred career levels.

In [ ]:
# Group by company to get aggregate metrics
comp_seg = df.groupby('company').agg(
    total_jobs=('title', 'count')
).reset_index()

# Merge with career level ratios
level_dummies = pd.get_dummies(df['career_level'])
comp_levels = pd.concat([df['company'], level_dummies], axis=1).groupby('company').sum()
comp_seg = comp_seg.merge(comp_levels, on='company')

# Filter out companies with too few jobs to segment reliably
comp_seg_top = comp_seg[comp_seg['total_jobs'] >= 3].copy()

if len(comp_seg_top) > 5:
    # Normalize features for clustering
    features = comp_seg_top.drop('company', axis=1)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    
    kmeans_comp = KMeans(n_clusters=3, random_state=42, n_init=10)
    comp_seg_top['segment'] = kmeans_comp.fit_predict(scaled_features)
    
    # Visualize
    y_col = comp_seg_top.columns[2] if len(comp_seg_top.columns) > 2 else 'total_jobs'
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='total_jobs', y=y_col, hue='segment', \
                    size='total_jobs', sizes=(50, 400), palette='viridis', \
                    data=comp_seg_top, alpha=0.8)
    plt.title(f'Company Segmentation (Hiring Volume vs {y_col})', fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig('output/company_segments.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Company Segment Sample:")
    display(comp_seg_top[['company', 'total_jobs', 'segment']].head(10))
else:
    print('⚠️ Not enough high-volume companies for robust segmentation.')